In [1]:
!pip install spotipy python-dotenv python-docx

In [2]:
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyOAuth

env_path = r"C:\Users\naiar\OneDrive\Documents\GitHub\time-capsule\.env.txt"
print("File exists:", os.path.exists(env_path))

load_dotenv(env_path)
print("CLIENT_ID:", os.environ.get("SPOTIPY_CLIENT_ID"))

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    scope="playlist-modify-private playlist-modify-public playlist-read-private"
))

me = sp.current_user()
print(f"Logged in as: {me['display_name']}")

File exists: True
CLIENT_ID: 02dabb9aced748f89676a98ba10e8369
Logged in as: naiara.soares


In [3]:
def search_track(sp, song, artist):
  """ 
  Looks up a song on Spotify. Returns the track URI if it finds a match, or None if not.
  """
    
    query = f"track:{song} artist:{artist}"
    results = sp.search(q=query, type="track", limit=1)
    items = results["tracks"]["items"]

    if not items:
        return None

    track = items[0]
    print(f"Found: {track['name']} - {track['artists'][0]['name']}")
    return track["uri"]

In [4]:
uri = search_track(sp, "Yellow", "Coldplay")
print(uri)

Found: Yellow - Coldplay
spotify:track:4oQ9xRKqZBpiGecr7Ji2Hi


In [5]:
def find_or_create_playlist(sp, playlist_name="Time Capsule"):
  """
    Checks if I already have a "Time Capsule" playlist and reuses it if so.
    Only makes a new one if it really doesn't exist yet - don't want duplicates.
    """
    
    offset = 0
    while True:
        response = sp.current_user_playlists(limit=50, offset=offset)
        playlists = response["items"]

        for playlist in playlists:
            if playlist["name"] == playlist_name:
                print(f"Found existing playlist: {playlist_name}")
                return playlist["id"]

        if response["next"] is None:
            break
        offset += 50

    print(f"No existing '{playlist_name}' playlist found - creating a new one.")
    new_playlist = sp.current_user_playlist_create(
        name=playlist_name,
        public=False,
        description="A musical time capsule - one memory at a time."
    )
    return new_playlist["id"]

In [6]:
def add_to_time_capsule(sp, song, artist):
    """
    Searches for the song and adds it to the Time Capsule playlist.
    Returns True if it was added successfully, False otherwise.
    """
    
    track_uri = search_track(sp, song, artist)

    if track_uri is None:
        print("Song not found on Spotify.")
        return False

    playlist_id = find_or_create_playlist(sp)
    sp.playlist_add_items(playlist_id, [track_uri])
    return True

In [7]:
try:
    playlist_id = find_or_create_playlist(sp)
    print("Playlist ID:", playlist_id)
except Exception as e:
    print("ERROR:", e)

Found existing playlist: Time Capsule
Playlist ID: 6XYY2p2IRfo5R2FYXsdSnt


In [18]:
import os
from datetime import date
from docx import Document
import shutil

PROJECT_FOLDER = r"C:\Users\naiar\OneDrive\Documents\GitHub\time-capsule"
LOGS_FOLDER = os.path.join(PROJECT_FOLDER, "logs")

def get_next_edition_number(logs_folder=LOGS_FOLDER):
    if not os.path.isdir(logs_folder):
        return 1
    existing = [f for f in os.listdir(logs_folder) if f.endswith(".docx")]
    return len(existing) + 1
"""counts how many memories are already saved so each new one gets its own "edition" number
"""


"""Below, builds the Word doc for one memory - song, artist, date, and the memory itself. Returns the file path."""
def create_memory_document(song, artist, memory_text):
    edition_number = get_next_edition_number()
    today_str = date.today().isoformat()
    doc = Document()
    doc.add_heading("Time Capsule", level=0)
    doc.add_heading(f"Memory No. {edition_number:03d} - Limited Edition", level=1)
    doc.add_paragraph(f"Date sealed: {today_str}")
    doc.add_paragraph(f"Song: {song}")
    doc.add_paragraph(f"Artist: {artist}")
    doc.add_paragraph("")
    doc.add_heading("The Memory", level=2)
    doc.add_paragraph(memory_text)
    filename = f"TimeCapsule_{today_str}_Ed{edition_number:03d}.docx"
    filepath = os.path.join(PROJECT_FOLDER, filename)
    doc.save(filepath)
    return filepath


 """Below, moves the finished document into the logs folder (makes the folder first if it's not there yet)."""
def move_to_logs(filepath, logs_folder=LOGS_FOLDER):
    os.makedirs(logs_folder, exist_ok=True)
    filename = os.path.basename(filepath)
    new_path = os.path.join(logs_folder, filename)
    shutil.move(filepath, new_path)
    return new_path

In [9]:
from docx import Document

def search_memories(keyword, logs_folder=LOGS_FOLDER):
  """
    Goes through everything saved in logs and checks if the keyword shows up
    in the song, artist, or memory text. Leave it blank to just list everything.
    """
    results = []
    if not os.path.isdir(logs_folder):
        print("No memories archived yet.")
        return results
    keyword_lower = keyword.lower().strip()
    for filename in sorted(os.listdir(logs_folder)):
        if not filename.endswith(".docx"):
            continue
        filepath = os.path.join(logs_folder, filename)
        doc = Document(filepath)
        full_text = "\n".join(paragraph.text for paragraph in doc.paragraphs)
        if keyword_lower == "" or keyword_lower in full_text.lower():
            results.append({"filename": filename, "content": full_text})
    return results

## song = input("Viajando Por El Mundo")
artist = input("Karol G, Manu Chao")
memory_text = input("This was my first solo song time trip by my own")

filepath = create_memory_document(song, artist, memory_text)
print(f"Saved: {filepath}")

In [11]:
new_path = move_to_logs(filepath)
print(f"Moved to: {new_path}")

Moved to: C:\Users\naiar\OneDrive\Documents\GitHub\time-capsule\logs\TimeCapsule_2026-08-13_Ed008.docx


In [12]:
def add_memory():
   """
    This is the main flow: ask the user for the song + memory, save it as a Word
    doc, try to add the song on Spotify, and only move the file into logs if that
    actually worked.
    """
    song = input("Song title: ")
    artist = input("Artist: ")
    memory_text = input("Your memory: ")

    filepath = create_memory_document(song, artist, memory_text)
    print(f"Memory saved as: {filepath}")

    success = add_to_time_capsule(sp, song, artist)

    if success:
        new_path = move_to_logs(filepath)
        print(f"Song added to Spotify! Memory archived at: {new_path}")
    else:
        print(f"Song not added to Spotify. Memory stays at: {filepath} (not archived)")

## add_memory()


In [13]:
found = search_memories("")

print(f"Found {len(found)} memories:\n")
for memory in found:
    print("-" * 40)
    print(memory["filename"])
    print(memory["content"])
    print()

Found 8 memories:

----------------------------------------
TimeCapsule_2026-08-12.docx
Time Capsule
Date sealed: 2026-08-12
Song: Murder on the dancefloor
Artist: Sophie Ellis

The Memory
Enjoying the nigth with friends

----------------------------------------
TimeCapsule_2026-08-12_Ed001.docx
Time Capsule
Memory No. 001 - Limited Edition
Date sealed: 2026-08-12
Song: Golden
Artist: Harry Styles

The Memory
Holidays in Rio de Janeiro

----------------------------------------
TimeCapsule_2026-08-12_Ed003.docx
Time Capsule
Memory No. 003 - Limited Edition
Date sealed: 2026-08-12
Song: 
Artist: 

The Memory


----------------------------------------
TimeCapsule_2026-08-12_Ed004.docx
Time Capsule
Memory No. 004 - Limited Edition
Date sealed: 2026-08-12
Song: Home
Artist: Edith Whiskers

The Memory
I remember my mom

----------------------------------------
TimeCapsule_2026-08-13_Ed005.docx
Time Capsule
Memory No. 005 - Limited Edition
Date sealed: 2026-08-13
Song: 
Artist: 

The Memory



In [14]:
import subprocess

result = subprocess.run(
    ["git", "status"],
    cwd=r"C:\Users\naiar\OneDrive\Documents\GitHub\time-capsule",
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	logs/TimeCapsule_2026-08-13_Ed005.docx
	logs/TimeCapsule_2026-08-13_Ed006.docx
	logs/TimeCapsule_2026-08-13_Ed007.docx
	logs/TimeCapsule_2026-08-13_Ed008.docx

nothing added to commit but untracked files present (use "git add" to track)




In [15]:
def show_menu():
    print("\nWhat would you like to do?")
    print("1. Seal a new memory")
    print("2. Search past memories")
    print("3. Exit")

def main():
    print("=" * 50)
    print("TIME CAPSULE - Music and memories")
    print("=" * 50)
    print("Every memory you seal here is special moment to remember,")
    print("a limited edition, organized and magical,")
    print("you can save your memores chooosing a song, dated and archived forever.")

    while True:
        show_menu()
        choice = input("Choose an option (1-3): ").strip()

        if choice == "1":
            add_memory()
            
        elif choice == "2":
            keyword = input("Search by song, artist, or word (leave blank for all): ")
            found = search_memories(keyword)
            print(f"\nFound {len(found)} memories:\n")
            for memory in found:
                print("-" * 40)
                print(memory["filename"])
                print(memory["content"])
                
        elif choice == "3":
            print("\nYour Time Capsule is sealed. See you next time.")
            break

    else:
        print("Invalid option. Please choose 1, 2 or 3.")
        
        
            
    

In [16]:
main()

TIME CAPSULE - Music and memories
Every memory you seal here is special moment to remember,
a limited edition, organized and magical,
you can save your memores chooosing a song, dated and archived forever.

What would you like to do?
1. Seal a new memory
2. Search past memories
3. Exit


Choose an option (1-3):  2
Search by song, artist, or word (leave blank for all):  Gabriela



Found 1 memories:

----------------------------------------
TimeCapsule_2026-08-13_Ed007.docx
Time Capsule
Memory No. 007 - Limited Edition
Date sealed: 2026-08-13
Song: Yellow
Artist: Coldplay

The Memory
I remember my friend Gabriela in her special wedding day

What would you like to do?
1. Seal a new memory
2. Search past memories
3. Exit


Choose an option (1-3):  3



Your Time Capsule is sealed. See you next time.


In [17]:
playlists = sp.current_user_playlists()["items"]
for p in playlists:
    if "Time Capsule" in p["name"]:
        total = p.get("tracks", {}).get("total", "?")
        print(p["id"], "-", p["name"], "-", total, "tracks")

6XYY2p2IRfo5R2FYXsdSnt - Time Capsule - ? tracks


In [19]:
import subprocess

result = subprocess.run(
    ["git", "log", "--oneline", "--all"],
    cwd=r"C:\Users\naiar\OneDrive\Documents\GitHub\time-capsule",
    capture_output=True,
    text=True
)
print(result.stdout)

98d8df1 Add limited edition numbering, fix folder paths, add gitignore
adede68 Initial commit

